# 01 · Data Collection & Processing
**Project**: War Shocks & Global Financial Risk Spillover  
**Period**: 2006-01-01 → 2025-12-31  
**Output**: `data/processed/` — all_variables_aligned, log_returns, rolling_volatility, war_dummies

All data collection and processing logic lives in `src/data_loader.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data_loader import (
    load_yahoo_equity, load_csi300, load_yahoo_controls, load_fred,
    merge_and_align, compute_returns, compute_volatility,
    build_war_dummies, save,
)

START = "2006-01-01"
END   = "2025-12-31"

## Step 1 · Load Equity Indices

In [ ]:
print("=" * 55)
print("Yahoo Finance — Equity Indices")
print("=" * 55)
equity_frames = load_yahoo_equity(START, END)

print()
print("CSI 300 — Local Excel")
print("-" * 55)
csi300 = load_csi300(START, END)

## Step 2 · Load Control Variables

In [ ]:
print("=" * 55)
print("Yahoo Finance — Safe-haven & Control Variables")
print("=" * 55)
control_frames = load_yahoo_controls(START, END)

## Step 3 · Load FRED Macro Variables

In [ ]:
print("=" * 55)
print("FRED — Local Excel")
print("=" * 55)
fred_df = load_fred(START, END)

## Step 4 · Merge & Align to S&P 500 Trading Days

In [ ]:
print("=" * 55)
print("Merging & Aligning")
print("=" * 55)
combined = merge_and_align(equity_frames, control_frames, fred_df, csi300)
print(f"\nFinal shape : {combined.shape}")
print(f"Date range  : {combined.index[0].date()} ~ {combined.index[-1].date()}")
combined.head(3)

## Step 5 · Compute Returns & Rolling Volatility

In [ ]:
returns_df = compute_returns(combined)
vol_df     = compute_volatility(returns_df, window=21)

print(f"Returns    shape : {returns_df.shape}")
print(f"Volatility shape : {vol_df.shape}")
print("\nReturns (equity only) — descriptive stats:")
equity_ret_cols = [c for c in returns_df.columns if c.endswith("_ret") and "Gold" not in c and "DXY" not in c and "Silver" not in c and "Brent" not in c and "WTI" not in c]
returns_df[equity_ret_cols].describe().round(6)

## Step 6 · Build War Dummy Variables

In [ ]:
print("=" * 55)
print("War Event Dummies")
print("=" * 55)
war_dummy = build_war_dummies(combined.index, END)
war_dummy.value_counts().sort_index()

## Step 7 · Save to `data/processed/`

In [ ]:
print("Saving processed files...")
save(combined,    "all_variables_aligned.xlsx")
save(returns_df,  "log_returns.xlsx")
save(vol_df,      "rolling_volatility.xlsx")
save(war_dummy,   "war_dummies.xlsx")
print("\nAll files saved to data/processed/")

## Step 8 · Quick Visual Check

In [ ]:
# Plot: rolling volatility of all equity indices
fig, ax = plt.subplots(figsize=(14, 4))
for col in vol_df.columns:
    ax.plot(vol_df.index, vol_df[col], linewidth=0.8, alpha=0.75, label=col.replace("_vol", ""))

# Shade war periods
from src.data_loader import DATA_RAW_DIR
import pandas as pd
events = pd.read_excel(f"{DATA_RAW_DIR}/war_events.xlsx", parse_dates=["start_date","end_date"])
for _, row in events.iterrows():
    end = row["end_date"] if pd.notna(row["end_date"]) else pd.Timestamp(END)
    color = "#d62728" if row["event_name"] in ["Israel-Lebanon 2006","Gaza Cast Lead 2008","Israel-Hamas 2023","Israel-Iran 2024"] else "#ff7f0e"
    ax.axvspan(row["start_date"], end, alpha=0.12, color=color, label=row["event_name"])

ax.set_title("Annualised 21-Day Rolling Volatility — All Equity Indices", fontsize=12)
ax.set_ylabel("Volatility (annualised)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=7, ncol=4, loc="upper right")
plt.tight_layout()
plt.savefig("../outputs/figures/vol_timeseries_check.png", dpi=150)
plt.show()

---
**Next** → `02_eda.ipynb` for exploratory analysis  
**Next** → `03_spillover_index.ipynb` to build the Diebold-Yilmaz spillover index